# 9. GCN subtype training and analysis

Predicts `subpose_label` using the same 76 features and fixed data splits as notebooks 7 and 8. Run those notebooks first to create the six baseline subtype results.

Architecture: 23 nodes × XYZ, two graph-convolution layers (64 channels), ReLU and dropout, mean pooling, concatenate seven angles, linear head with nine subtype outputs. The fixed graph includes custom nose-to-shoulder links. Normalization is fit only on training and saved inside the model.

Checkpoint selection: validation macro-F1, with accuracy as a tie-breaker; no test-driven tuning. Full test evaluation includes four samples from two unseen subtypes and uses 11 labels. Known-subtype test metrics separately cover 461 samples and nine labels. These macro-F1 scores therefore use different class sets.

Run all cells. Uses project-local CPU PyTorch 2.5.1 and code/gcn_model.py for the shared architecture and backend loader. Saves a .pt state dictionary, JSON metadata, prediction and metric CSVs, confusion matrices, training curves and analysis.


## Setup

In [1]:
from pathlib import Path
import sys
import json
import time
import re
import random
import hashlib
import importlib.metadata

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / 'csv_data/prepared_to_train/keypoints_train.csv').is_file())
sys.path.insert(0, str(PROJECT_ROOT / '.dependencies'))
sys.path.insert(0, str(PROJECT_ROOT / 'code'))
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from gcn_model import PoseGCN, load_gcn, predict_poses

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
TARGET_COLUMN, SAMPLE_COLUMN = 'subpose_label', 'image_name'
SELECTION_METRIC = 'macro_f1'
INPUT_DIR = PROJECT_ROOT / 'csv_data/prepared_to_train'
INPUT_FILES = {'train': INPUT_DIR / 'keypoints_train.csv',
               'validation': INPUT_DIR / 'keypoints_validation.csv',
               'test': INPUT_DIR / 'keypoints_test_dbscan_subposes.csv'}
OUTPUT_DIR = PROJECT_ROOT / 'model/code9_gcn_subpose_classification'
MODEL_DIR = OUTPUT_DIR / 'models'
PREDICTION_DIR = OUTPUT_DIR / 'predictions'
CONFUSION_DIR = OUTPUT_DIR / 'confusion_matrices'
for directory in [OUTPUT_DIR, MODEL_DIR, PREDICTION_DIR, CONFUSION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
BASELINE_DIR = PROJECT_ROOT / 'model/code7_subpose_classification'


## Load and validate splits and baseline results

In [2]:
datasets = {split: pd.read_csv(path) for split, path in INPUT_FILES.items()}
FEATURE_COLUMNS = [
    column for column in datasets['train'].columns
    if column.startswith('kp_') or column.endswith('_angle_deg')
]
if not FEATURE_COLUMNS:
    raise ValueError('No keypoint or angle feature columns were found.')

train_columns = set(datasets['train'].columns)
for split, dataframe in datasets.items():
    missing = {SAMPLE_COLUMN, TARGET_COLUMN, *FEATURE_COLUMNS}.difference(dataframe.columns)
    if missing:
        raise ValueError(f'{split} is missing columns: {sorted(missing)}')
    if dataframe[[SAMPLE_COLUMN, TARGET_COLUMN]].isna().any().any():
        raise ValueError(f'{split} has missing sample IDs or subtype labels.')
    values = dataframe[FEATURE_COLUMNS].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError(f'{split} feature data contains NaN or infinite values.')
    if dataframe[SAMPLE_COLUMN].duplicated().any():
        raise ValueError(f'{split} contains duplicate sample paths.')

CLASSES = sorted(datasets['train'][TARGET_COLUMN].astype(str).unique())
for split in ('validation', 'test'):
    unknown = set(datasets[split][TARGET_COLUMN].astype(str)).difference(CLASSES)
    if unknown:
        if split == 'validation':
            raise ValueError(f'Validation contains subtypes absent from training: {sorted(unknown)}')
        print(f'{split}: unseen training subtypes retained for evaluation: {sorted(unknown)}')

X = {split: df[FEATURE_COLUMNS] for split, df in datasets.items()}
y = {split: df[TARGET_COLUMN].astype(str) for split, df in datasets.items()}
pd.DataFrame({
    split: y[split].value_counts().reindex(CLASSES, fill_value=0)
    for split in datasets
}).rename_axis('pose')


for first, second in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    if set(datasets[first][SAMPLE_COLUMN]) & set(datasets[second][SAMPLE_COLUMN]):
        raise ValueError(f'Sample overlap between {first} and {second}')
EVALUATION_CLASSES = {split: sorted(set(CLASSES) | set(df[TARGET_COLUMN].astype(str)))
                      for split, df in datasets.items()}
coverage = pd.DataFrame({split: df[TARGET_COLUMN].value_counts()
                         for split, df in datasets.items()}).fillna(0).astype(int)
coverage['seen_in_training'] = coverage.index.isin(CLASSES)
coverage.rename_axis('subpose').to_csv(OUTPUT_DIR / 'subtype_split_coverage.csv')
unseen_tables = []
for split, df in datasets.items():
    unseen = df.loc[~df[TARGET_COLUMN].isin(CLASSES), [SAMPLE_COLUMN, 'label', TARGET_COLUMN]].copy()
    unseen.insert(0, 'split', split)
    unseen_tables.append(unseen)
pd.concat(unseen_tables).to_csv(OUTPUT_DIR / 'unseen_subtype_samples.csv', index=False)
display(coverage)

encoder = LabelEncoder().fit(CLASSES)
y_encoded = encoder.transform(y['train'])
for first, second in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    if set(datasets[first][SAMPLE_COLUMN]) & set(datasets[second][SAMPLE_COLUMN]):
        raise ValueError(f'Sample overlap between {first} and {second}')
baseline_manifest = json.loads((BASELINE_DIR / 'model_manifest.json').read_text(encoding='utf-8'))
if baseline_manifest['target_column'] != TARGET_COLUMN:
    raise ValueError('Run notebook 7 for subtype classification first.')
for split, path in INPUT_FILES.items():
    if baseline_manifest['input_sha256'][split] != hashlib.sha256(path.read_bytes()).hexdigest():
        raise ValueError('Baseline input data differs from current data.')
if baseline_manifest['feature_columns'] != FEATURE_COLUMNS or baseline_manifest['classes'] != CLASSES:
    raise ValueError('Baseline features/classes differ from the current data.')
baseline_metrics = pd.read_csv(BASELINE_DIR / 'overall_metrics.csv')
for row in baseline_metrics.itertuples():
    previous = pd.read_csv(BASELINE_DIR / 'predictions' / f'{row.model}_{row.split}_predictions.csv')
    current = datasets[row.split].set_index(SAMPLE_COLUMN)[TARGET_COLUMN].astype(str)
    if previous['sample'].duplicated().any() or set(previous['sample']) != set(current.index):
        raise ValueError('Baseline evaluation samples differ from current samples.')
    if not np.array_equal(previous['true_label'].astype(str), current.loc[previous['sample']].to_numpy()):
        raise ValueError('Baseline evaluation labels differ from current labels.')
    measured_f1 = precision_recall_fscore_support(previous.true_label, previous.predicted_label,
        labels=EVALUATION_CLASSES[row.split], average='macro', zero_division=0)[2]
    if not np.isclose(measured_f1, row.macro_f1):
        raise ValueError('Baseline metrics do not match saved predictions.')

boosting_dir = PROJECT_ROOT / 'model/code8_boosting_subpose_classification'
boost_manifest = json.loads((boosting_dir / 'model_manifest.json').read_text())
assert boost_manifest['target_column'] == TARGET_COLUMN
assert boost_manifest['classes'] == CLASSES
assert boost_manifest['feature_columns'] == FEATURE_COLUMNS
assert boost_manifest['evaluation_classes'] == EVALUATION_CLASSES
for split, path in INPUT_FILES.items():
    assert boost_manifest['input_sha256'][split] == hashlib.sha256(path.read_bytes()).hexdigest()
boost_metrics = pd.read_csv(boosting_dir / 'overall_metrics.csv')
for row in boost_metrics.itertuples():
    previous = pd.read_csv(boosting_dir / 'predictions' / f'{row.model}_{row.split}_predictions.csv')
    current = datasets[row.split].set_index(SAMPLE_COLUMN)[TARGET_COLUMN].astype(str)
    assert not previous['sample'].duplicated().any()
    assert set(previous['sample']) == set(current.index)
    assert np.array_equal(previous.true_label, current.loc[previous['sample']].to_numpy())
    score = precision_recall_fscore_support(previous.true_label, previous.predicted_label,
              labels=EVALUATION_CLASSES[row.split], average='macro', zero_division=0)[2]
    assert np.isclose(score, row.macro_f1)
baseline_metrics = pd.concat([baseline_metrics, boost_metrics], ignore_index=True)
baseline_pose_metrics = pd.concat([pd.read_csv(d / 'classification_report_per_subpose.csv')
                                  for d in [BASELINE_DIR, boosting_dir]], ignore_index=True)
baseline_known_metrics = pd.concat([pd.read_csv(d / 'known_subtype_test_metrics.csv')
                                   for d in [BASELINE_DIR, boosting_dir]], ignore_index=True)


test: unseen training subtypes retained for evaluation: ['goddess_subpose_3', 'plank_subpose_3']
                          train  validation  test  seen_in_training
subpose_label                                                      
downdog_subpose_1           159          40    94              True
goddess_subpose_2           130          32    78              True
goddess_subpose_3             0           0     2             False
plank_subpose_1              10           3     1              True
plank_subpose_2              99          25    44              True
plank_subpose_3               0           0     2             False
plank_subpose_4              95          23    68              True
tree_left_subpose_2          40          10    29              True
tree_right_subpose_2         80          20    40              True
warrior2_left_subpose_1      70          17    42              True
warrior2_right_subpose_1    129          33    65              True


## Build the skeleton graph

In [3]:
NODE_IDS = [0, *range(11, 33)]
ANGLE_COLUMNS = [c for c in FEATURE_COLUMNS if c.endswith('_angle_deg')]
expected_features = [f'kp_{node}_{axis}' for node in NODE_IDS for axis in 'xyz'] + ANGLE_COLUMNS
assert FEATURE_COLUMNS == expected_features, 'Unexpected feature order.'
# Undirected body connections. Nose-to-shoulder edges are explicit custom links
# because the intermediate facial landmarks were removed from this dataset.
EDGES = [(0, 11), (0, 12), (11, 12), (11, 13), (13, 15),
         (15, 17), (15, 19), (15, 21), (17, 19), (12, 14), (14, 16),
         (16, 18), (16, 20), (16, 22), (18, 20), (11, 23), (12, 24),
         (23, 24), (23, 25), (24, 26), (25, 27), (26, 28),
         (27, 29), (28, 30), (29, 31), (30, 32), (27, 31), (28, 32)]
config = {'node_ids': NODE_IDS, 'edges': EDGES, 'self_loops': True,
          'adjacency_normalization': 'D^-1/2 (A+I) D^-1/2',
          'hidden_channels': 64, 'dropout': 0.1, 'feature_count': len(FEATURE_COLUMNS),
          'angle_count': len(ANGLE_COLUMNS), 'class_count': len(CLASSES),
          'pooling': 'mean', 'graph_layers': 2}
model = PoseGCN(config)
train_values = X['train'].to_numpy(dtype=np.float32)
mean, scale = train_values.mean(axis=0), train_values.std(axis=0)
scale[scale < 1e-8] = 1.0
model.feature_mean.copy_(torch.from_numpy(mean))
model.feature_scale.copy_(torch.from_numpy(scale))
# Only training and validation enter the training loop.
train_x = torch.from_numpy(train_values)
train_y = torch.from_numpy(y_encoded.astype(np.int64))
validation_x = torch.tensor(X['validation'].to_numpy(dtype=np.float32))
validation_y = encoder.transform(y['validation']).astype(np.int64)
TRAINING = {'max_epochs': 600, 'patience': 100, 'batch_size': 64,
            'learning_rate': 0.001, 'weight_decay': 0.0001, 'optimizer': 'Adam',
            'loss': 'unweighted cross entropy', 'seed': RANDOM_STATE, 'device': 'cpu'}
print(f'Graph: {len(NODE_IDS)} nodes, {len(EDGES)} undirected edges plus self-loops.')


Graph: 23 nodes, 28 undirected edges plus self-loops.


## Train and select the checkpoint

In [4]:
optimizer = torch.optim.Adam(model.parameters(), lr=TRAINING['learning_rate'],
                             weight_decay=TRAINING['weight_decay'])
criterion = torch.nn.CrossEntropyLoss()
best_score = (-1.0, -1.0)
best_epoch, stale_epochs = 0, 0
history = []
started = time.perf_counter()
for epoch in range(1, TRAINING['max_epochs'] + 1):
    model.train()
    order = torch.randperm(len(train_x))
    running_loss = 0.0
    for indices in order.split(TRAINING['batch_size']):
        optimizer.zero_grad()
        logits = model(train_x[indices])
        loss = criterion(logits, train_y[indices])
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(indices)
    model.eval()
    with torch.inference_mode():
        logits = model(validation_x)
        predictions = logits.argmax(1).numpy()
        validation_loss = criterion(logits, torch.from_numpy(validation_y)).item()
    f1 = precision_recall_fscore_support(validation_y, predictions,
        labels=np.arange(len(CLASSES)), average='macro', zero_division=0)[2]
    accuracy = accuracy_score(validation_y, predictions)
    history.append({'epoch': epoch, 'train_loss': running_loss / len(train_x),
                    'validation_loss': validation_loss, 'validation_macro_f1': f1,
                    'validation_accuracy': accuracy})
    if (f1, accuracy) > best_score:
        best_score, best_epoch, stale_epochs = (f1, accuracy), epoch, 0
        torch.save(model.state_dict(), MODEL_DIR / 'gcn.pt')
    else:
        stale_epochs += 1
    if epoch == 1 or epoch % 25 == 0:
        print(f'Epoch {epoch}: validation macro-F1={f1:.4f}; best={best_score[0]:.4f}', flush=True)
    if stale_epochs >= TRAINING['patience']:
        print(f'Early stopping at epoch {epoch}.', flush=True)
        break
fit_seconds = time.perf_counter() - started
model.load_state_dict(torch.load(MODEL_DIR / 'gcn.pt', map_location='cpu', weights_only=True))
model.eval()
history_table = pd.DataFrame(history)
history_table.to_csv(OUTPUT_DIR / 'training_history.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history_table.plot(x='epoch', y=['train_loss', 'validation_loss'], ax=axes[0])
history_table.plot(x='epoch', y=['validation_macro_f1', 'validation_accuracy'], ax=axes[1])
for ax in axes:
    ax.axvline(best_epoch, linestyle='--', color='gray', label='Selected epoch')
    ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=180)
plt.close(fig)
print(f'Selected epoch: {best_epoch}; training time: {fit_seconds:.1f}s')

class EvaluationAdapter:
    classes_ = np.arange(len(CLASSES))

    def predict_proba(self, features):
        with torch.inference_mode():
            return model(torch.tensor(features.to_numpy(dtype=np.float32))).softmax(1).numpy()

    def predict(self, features):
        return self.predict_proba(features).argmax(1)

models = {'gcn': EvaluationAdapter()}


Epoch 1: validation macro-F1=0.1546; best=0.1546
Epoch 25: validation macro-F1=0.9001; best=0.9001
Epoch 50: validation macro-F1=0.9115; best=0.9185
Epoch 75: validation macro-F1=0.9217; best=0.9217
Epoch 100: validation macro-F1=0.9177; best=0.9217
Epoch 125: validation macro-F1=0.9235; best=0.9276
Epoch 150: validation macro-F1=0.9196; best=0.9276
Epoch 175: validation macro-F1=0.9269; best=0.9389
Epoch 200: validation macro-F1=0.9269; best=0.9389
Epoch 225: validation macro-F1=0.9331; best=0.9389
Epoch 250: validation macro-F1=0.9266; best=0.9389
Early stopping at epoch 260.
Selected epoch: 160; training time: 12.1s


## Evaluate the selected model

In [5]:
def safe_name(value: str) -> str:
    return re.sub(r'[^0-9A-Za-z_-]+', '_', str(value)).strip('_')


def aligned_probabilities(model, features: pd.DataFrame) -> np.ndarray:
    raw = model.predict_proba(features)
    model_classes = encoder.inverse_transform(model.classes_.astype(int)).tolist()
    indices = [model_classes.index(pose) for pose in CLASSES]
    return raw[:, indices]


def save_confusion_figure(matrix: np.ndarray, title: str, path: Path, labels) -> None:
    fig, ax = plt.subplots(figsize=(14, 12))
    image = ax.imshow(matrix, cmap='Blues')
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    ax.set(xticks=range(len(labels)), yticks=range(len(labels)),
           xticklabels=labels, yticklabels=labels,
           xlabel='Predicted subtype', ylabel='True subtype', title=title)
    plt.setp(ax.get_xticklabels(), rotation=35, ha='right')
    threshold = matrix.max() / 2 if matrix.size else 0
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            ax.text(column, row, int(matrix[row, column]), ha='center', va='center',
                    color='white' if matrix[row, column] > threshold else 'black')
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches='tight')
    plt.close(fig)


overall_rows = []
pose_rows = []
confusion_rows = []
fit_rows = []
known_rows = []

for model_name, estimator in models.items():
    model_path = MODEL_DIR / 'gcn.pt'
    fit_rows.append({'model': model_name, 'fit_seconds': fit_seconds, 'model_file': str(model_path)})

    for split in ('validation', 'test'):
        evaluation_labels = EVALUATION_CLASSES[split]
        predicted = encoder.inverse_transform(estimator.predict(X[split]).astype(int))
        probabilities = aligned_probabilities(estimator, X[split])
        precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
            y[split], predicted, labels=evaluation_labels, average='macro', zero_division=0
        )
        precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
            y[split], predicted, labels=evaluation_labels, average='weighted', zero_division=0
        )
        overall_rows.append({
            'model': model_name, 'split': split, 'samples': len(y[split]), 'evaluated_class_count': len(evaluation_labels),
            'unseen_subtype_samples': int((~y[split].isin(CLASSES)).sum()),
            'accuracy': accuracy_score(y[split], predicted),
            'macro_precision': precision_macro, 'macro_recall': recall_macro,
            'macro_f1': f1_macro, 'weighted_precision': precision_weighted,
            'weighted_recall': recall_weighted, 'weighted_f1': f1_weighted,
        })

        report = classification_report(
            y[split], predicted, labels=evaluation_labels, target_names=evaluation_labels,
            output_dict=True, zero_division=0,
        )
        for pose in evaluation_labels:
            pose_rows.append({
                'model': model_name, 'split': split, 'subpose': pose, 'seen_in_training': pose in CLASSES,
                'precision': report[pose]['precision'], 'recall': report[pose]['recall'],
                'f1': report[pose]['f1-score'], 'support': int(report[pose]['support']),
            })

        matrix = confusion_matrix(y[split], predicted, labels=evaluation_labels)
        pd.DataFrame(matrix, index=evaluation_labels, columns=evaluation_labels).rename_axis('true_subpose').to_csv(CONFUSION_DIR / f'{model_name}_{split}_confusion_matrix.csv')
        for true_index, true_pose in enumerate(evaluation_labels):
            for predicted_index, predicted_pose in enumerate(evaluation_labels):
                confusion_rows.append({
                    'model': model_name, 'split': split, 'true_subpose': true_pose,
                    'predicted_subpose': predicted_pose,
                    'count': int(matrix[true_index, predicted_index]),
                })
        save_confusion_figure(
            matrix, f'{model_name} - {split}',
            CONFUSION_DIR / f'{model_name}_{split}_confusion_matrix.png', evaluation_labels,
        )

        prediction_table = pd.DataFrame({
            'sample': datasets[split][SAMPLE_COLUMN].astype(str),
            'true_label': y[split], 'predicted_label': predicted,
            'is_correct': y[split].to_numpy() == predicted,
            'true_subtype_seen_in_training': y[split].isin(CLASSES),
            'main_pose': datasets[split]['label'],
        })
        for class_index, pose in enumerate(CLASSES):
            prediction_table[f'score_{safe_name(pose)}'] = probabilities[:, class_index]
        prediction_table.to_csv(
            PREDICTION_DIR / f'{model_name}_{split}_predictions.csv', index=False
        )

        if split == 'test':
            known_mask = y[split].isin(CLASSES).to_numpy()
            known_truth, known_predicted = y[split][known_mask], predicted[known_mask]
            kp, kr, kf, _ = precision_recall_fscore_support(known_truth, known_predicted,
                labels=CLASSES, average='macro', zero_division=0)
            known_rows.append({'model': model_name, 'split': 'test_known_subtypes',
                'samples': int(known_mask.sum()), 'excluded_unseen_samples': int((~known_mask).sum()),
                'accuracy': accuracy_score(known_truth, known_predicted),
                'macro_precision': kp, 'macro_recall': kr, 'macro_f1': kf})

overall_metrics = pd.DataFrame(overall_rows).sort_values(['split', 'macro_f1'], ascending=[True, False])
per_pose_metrics = pd.DataFrame(pose_rows).sort_values(['split', 'model', 'subpose'])
confusion_long = pd.DataFrame(confusion_rows)
training_times = pd.DataFrame(fit_rows).sort_values('fit_seconds')

overall_metrics.to_csv(OUTPUT_DIR / 'overall_metrics.csv', index=False)
per_pose_metrics.to_csv(OUTPUT_DIR / 'classification_report_per_subpose.csv', index=False)
confusion_long.to_csv(OUTPUT_DIR / 'confusion_matrices.csv', index=False)
training_times.to_csv(OUTPUT_DIR / 'training_times.csv', index=False)
display(overall_metrics)
display(per_pose_metrics)

pd.DataFrame(known_rows).to_csv(OUTPUT_DIR / 'known_subtype_test_metrics.csv', index=False)


  model       split  samples  ...  weighted_precision  weighted_recall  weighted_f1
1   gcn        test      465  ...            0.951988         0.959140     0.955181
0   gcn  validation      203  ...            0.951768         0.950739     0.950064

[2 rows x 12 columns]
   model       split                   subpose  ...    recall        f1  support
9    gcn        test         downdog_subpose_1  ...  0.968085  0.983784       94
10   gcn        test         goddess_subpose_2  ...  0.948718  0.948718       78
11   gcn        test         goddess_subpose_3  ...  0.000000  0.000000        2
12   gcn        test           plank_subpose_1  ...  1.000000  1.000000        1
13   gcn        test           plank_subpose_2  ...  0.954545  0.943820       44
14   gcn        test           plank_subpose_3  ...  0.000000  0.000000        2
15   gcn        test           plank_subpose_4  ...  0.955882  0.962963       68
16   gcn        test       tree_left_subpose_2  ...  0.965517  0.965517      

## Save metadata, verify reload and compare

In [6]:
manifest = {
    'best_model': 'gcn', 'best_model_file': 'models/gcn.pt',
    'selection_split': 'validation', 'selection_metric': 'macro_f1',
    'selection_value': best_score[0], 'best_epoch': best_epoch,
    'architecture': config, 'training': TRAINING,
    'feature_columns': FEATURE_COLUMNS, 'classes': CLASSES,
    'evaluation_classes': EVALUATION_CLASSES,
    'target_column': TARGET_COLUMN, 'sample_column': SAMPLE_COLUMN,
    'label_mapping_file': 'label_mapping.json',
    'preprocessing_file': 'preprocessing.json',
    'input_sha256': {s: hashlib.sha256(p.read_bytes()).hexdigest() for s, p in INPUT_FILES.items()},
    'versions': {'torch': torch.__version__, 'numpy': np.__version__, 'pandas': pd.__version__,
                 'scikit-learn': importlib.metadata.version('scikit-learn')},
    'note': 'Normalization fit on training only. Epoch selected using validation macro-F1, then accuracy. Test evaluated after selection. Single seeded run; no significance claim.',
}
(OUTPUT_DIR / 'model_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'label_mapping.json').write_text(json.dumps({'classes': CLASSES,
    'encoded_class_mapping': {str(i): c for i, c in enumerate(CLASSES)}}, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'preprocessing.json').write_text(json.dumps({'feature_columns': FEATURE_COLUMNS,
    'mean': mean.tolist(), 'scale': scale.tolist(), 'note': 'Also embedded in model state; do not apply twice.'}, indent=2), encoding='utf-8')
# Keep the architecture beside the weights for deployment.
import shutil
shutil.copyfile(PROJECT_ROOT / 'code/gcn_model.py', OUTPUT_DIR / 'gcn_model.py')
restored, restored_manifest = load_gcn(OUTPUT_DIR)
for split in ['validation', 'test']:
    labels, probabilities = predict_poses(restored, restored_manifest, X[split])
    np.testing.assert_allclose(probabilities, models['gcn'].predict_proba(X[split]), rtol=1e-6, atol=1e-8)
    saved = pd.read_csv(PREDICTION_DIR / f'gcn_{split}_predictions.csv')
    assert np.array_equal(labels, saved.predicted_label.to_numpy())
    np.testing.assert_allclose(probabilities, saved.filter(like='score_').to_numpy(), rtol=1e-6, atol=1e-8)
    assert np.allclose(probabilities.sum(1), 1.0)
    matrix = pd.read_csv(CONFUSION_DIR / f'gcn_{split}_confusion_matrix.csv', index_col=0)
    assert matrix.to_numpy().sum() == len(X[split])
combined = pd.concat([baseline_metrics, overall_metrics], ignore_index=True)
combined.to_csv(OUTPUT_DIR / 'comparison_all_models.csv', index=False)
pd.concat([baseline_pose_metrics, per_pose_metrics], ignore_index=True).to_csv(
    OUTPUT_DIR / 'comparison_per_subpose.csv', index=False)
ranking = combined[combined.split.eq('validation')].sort_values(['macro_f1', 'accuracy'], ascending=False)
ranking.to_csv(OUTPUT_DIR / 'comparison_validation_ranking.csv', index=False)
table = combined.pivot(index='model', columns='split', values=['accuracy', 'macro_f1']).reindex(ranking.model)
table.to_csv(OUTPUT_DIR / 'comparison_summary.csv')
best_baseline = baseline_metrics[baseline_metrics.split.eq('validation')].sort_values(
    ['macro_f1', 'accuracy'], ascending=False).iloc[0]['model']
deltas = []
for row in overall_metrics.to_dict('records'):
    reference = baseline_metrics[(baseline_metrics.model == best_baseline) & (baseline_metrics.split == row['split'])].iloc[0]
    deltas.append({'model': 'gcn', 'split': row['split'], 'baseline': best_baseline,
                  'accuracy_delta_percentage_points': 100 * (row['accuracy'] - reference.accuracy),
                  'macro_f1_delta_percentage_points': 100 * (row['macro_f1'] - reference.macro_f1)})
pd.DataFrame(deltas).to_csv(OUTPUT_DIR / 'comparison_vs_best_baseline.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric in zip(axes, ['accuracy', 'macro_f1']):
    table[metric].plot.bar(ax=ax, ylim=(0, 1.05), rot=35, title=metric)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'comparison_all_models.png', dpi=180, bbox_inches='tight')
plt.close(fig)
lines = ['# GCN comparison', '', '| Model | Validation accuracy | Validation macro-F1 | Test accuracy | Test macro-F1 |',
         '|---|---:|---:|---:|---:|']
for name in ranking.model:
    lines.append('| ' + name + ' | ' + ' | '.join(f'{table.loc[name, (metric, split)]:.4f}'
        for split in ['validation', 'test'] for metric in ['accuracy', 'macro_f1']) + ' |')
lines += ['', f'Selected GCN epoch: {best_epoch}. Overall validation winner: {ranking.iloc[0]["model"]}.',
          '', manifest['note'], '', 'Baseline evaluation sample IDs and labels were checked. Training, validation and test hashes match both baseline manifests.',
          '', 'Backend: import load_gcn and predict_poses from gcn_model.py; load_gcn(output_directory) restores the model and manifest. predict_poses(model, manifest, dataframe) returns labels and probabilities. Normalization is embedded in the model.']
(OUTPUT_DIR / 'comparison_report.md').write_text('\n'.join(lines), encoding='utf-8')
display(table)
print('Native checkpoint reload, predictions, probability and confusion matrix checks passed.')
print('Saved results:', OUTPUT_DIR)

known_comparison = pd.concat([baseline_known_metrics, pd.DataFrame(known_rows)], ignore_index=True)
known_comparison.to_csv(OUTPUT_DIR / 'comparison_known_subtype_test.csv', index=False)
with torch.inference_mode():
    train_predictions = model(train_x).argmax(1).numpy()
train_f1 = precision_recall_fscore_support(y_encoded, train_predictions,
    labels=np.arange(len(CLASSES)), average='macro', zero_division=0)[2]
pd.DataFrame([{'model': 'gcn', 'split': 'train', 'samples': len(train_x),
    'accuracy': accuracy_score(y_encoded, train_predictions), 'macro_f1': train_f1}]).to_csv(
    OUTPUT_DIR / 'training_fit_metrics.csv', index=False)
errors = pd.read_csv(PREDICTION_DIR / 'gcn_test_predictions.csv')
errors = errors[errors.true_label.ne(errors.predicted_label)]
error_pairs = errors.groupby(['true_label', 'predicted_label', 'true_subtype_seen_in_training']).size().reset_index(name='count').sort_values('count', ascending=False)
error_pairs.to_csv(OUTPUT_DIR / 'test_error_pairs.csv', index=False)
gcn_rank = ranking.model.tolist().index('gcn') + 1
gcn_test = overall_metrics[overall_metrics.split.eq('test')].iloc[0]
gcn_known = known_comparison[known_comparison.model.eq('gcn')].iloc[0]
gcn_validation = overall_metrics[overall_metrics.split.eq('validation')].iloc[0]
analysis = ['# GCN subtype analysis', '',
    f'GCN ranks {gcn_rank} of {len(ranking)} models by validation macro-F1. The validation winner is {ranking.iloc[0]["model"]}.', '',
    f'Selected epoch: {best_epoch}; completed epochs: {len(history)}; fit time: {fit_seconds:.1f} seconds.', '',
    '| Evaluation | Samples | Accuracy | Macro-F1 |', '|---|---:|---:|---:|',
    f'| Training (selected checkpoint) | {len(train_x)} | {accuracy_score(y_encoded, train_predictions):.4f} | {train_f1:.4f} |',
    f'| Validation (9 classes) | {len(validation_x)} | {gcn_validation.accuracy:.4f} | {gcn_validation.macro_f1:.4f} |',
    f'| Full test (11 classes) | {int(gcn_test.samples)} | {gcn_test.accuracy:.4f} | {gcn_test.macro_f1:.4f} |',
    f'| Known-subtype test (9 classes) | {int(gcn_known.samples)} | {gcn_known.accuracy:.4f} | {gcn_known.macro_f1:.4f} |', '',
    f'Compared with the validation-selected baseline ({best_baseline}), GCN validation macro-F1 differs by {deltas[1 if deltas[0]["split"] == "test" else 0]["macro_f1_delta_percentage_points"]:+.2f} percentage points.', '',
    'Full test includes goddess_subpose_3 and plank_subpose_3, with two samples each and no training examples. The nine-output classifier cannot predict these labels. They remain in the full confusion matrix and contribute zero per-class F1. The separate known-subtype evaluation is a different subset, not a replacement for the full test result.', '',
    f'At the selected checkpoint, training minus validation macro-F1 is {100*(train_f1-gcn_validation.macro_f1):.2f} percentage points. This describes the fit gap; one split and one seed do not establish statistical significance or the cause of a performance difference.', '',
    'Most frequent test errors:', '', '| True subtype | Predicted subtype | Count | Seen in training |', '|---|---|---:|---|']
for row in error_pairs.head(10).itertuples():
    analysis.append(f'| {row.true_label} | {row.predicted_label} | {row.count} | {row.true_subtype_seen_in_training} |')
analysis += ['', 'Interpretation and limits:', '',
    '- This is a compact two-layer GCN with mean pooling and angle features, not evidence about every GCN architecture.',
    '- Mean pooling discards node order at the readout. Preserving joint identity or a richer readout is a possible future experiment, not a demonstrated explanation of these errors.',
    '- Several subtypes have very small evaluation supports, so a few mistakes can substantially change macro-F1. Consult the per-subtype report.',
    '- Additional training examples are needed for the two unseen subtypes. Keep test samples held out.',
    '- Use validation for future configuration choices. No configuration was changed based on these test scores.', '',
    'See comparison_report.md for the seven-model table and comparison_known_subtype_test.csv for the common known-subtype subset.']
(OUTPUT_DIR / 'analysis.md').write_text('\n'.join(analysis), encoding='utf-8')
with (OUTPUT_DIR / 'comparison_report.md').open('a', encoding='utf-8') as report:
    report.write('\n\nFull test uses 11 labels, including two unseen training subtypes (four samples). See analysis.md and comparison_known_subtype_test.csv for interpretation and the separate nine-class test subset.\n')
display(pd.DataFrame(known_rows))
print('Analysis saved:', OUTPUT_DIR / 'analysis.md')


                     accuracy             macro_f1           
split                    test validation      test validation
model                                                        
random_forest        0.978495   0.965517  0.803965   0.950638
mlp                  0.974194   0.960591  0.800881   0.941524
lightgbm             0.982796   0.955665  0.807654   0.940823
gcn                  0.959140   0.950739  0.791083   0.938864
rbf_svm              0.978495   0.955665  0.804311   0.936540
xgboost              0.982796   0.950739  0.807654   0.936334
logistic_regression  0.965591   0.940887  0.793397   0.913660
Native checkpoint reload, predictions, probability and confusion matrix checks passed.
Saved results: C:\Users\vgohu\Desktop\yoga_project\model\code9_gcn_subpose_classification
  model                split  samples  ...  macro_precision  macro_recall  macro_f1
0   gcn  test_known_subtypes      461  ...         0.968606      0.972485  0.970138

[1 rows x 8 columns]
Analysis save